# Notebook 3 — Capacity KPI Forecasting
**Target KPI:** `throughput_dl_mbps`

| Model | Why |
|---|---|
| **LSTM** | Baseline — solid on throughput time-series |
| **CNN-LSTM** | CNN extracts local burst/ramp patterns, LSTM captures the trend on top |
| **GRU** | Lighter comparison baseline |

In [1]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
# ─────────────────────────────────────────────
#  CONFIGURATION  —  edit these values freely
# ─────────────────────────────────────────────

DATA_PATH   = "../Data/Model_data.csv"
OUTPUT_DIR  = "../outputs/capacity"

TARGET_KPI  = "throughput_dl_mbps"
TOLERANCE   = 5.0       # ± 5 Mbps

SEQ_LEN     = 12
TRAIN_RATIO = 0.8

HIDDEN_SIZE = 64
NUM_LAYERS  = 2
EPOCHS      = 20
BATCH_SIZE  = 2048
LR          = 1e-3

RNG_SEED = 42
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")

Device: cpu


In [4]:
print("Loading data...")
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head(3)

Loading data...
Rows: 210,528 | Columns: 20


,timestamp,slice_type,latitude,longitude,one_way_latency_ms,jitter_ms,rtt_ms,packet_delay_budget_ms,handover_interruption_time_ms,reliability_percent,packet_loss_percent,packet_loss_rate_percent,bler_percent,throughput_dl_mbps,throughput_ul_mbps,spectral_efficiency_bps_hz,handover_success_rate_percent,energy_efficiency_bits_per_joule,anomaly,anomaly_type
0,2024-01-01 00:00:00,URLLC,33.800386,-7.547638,2.5865,0.5029,5.3423,0.7614,5.2166,99.9995,0.0005,0.0005,0.4930,106.5463,103.6793,9.9301,99.5036,476587788.0,0,normal
1,2024-01-01 00:05:00,URLLC,33.802700,-7.553952,2.4543,0.4950,5.1841,0.7626,5.0939,99.9995,0.0005,0.0005,0.4954,102.3002,102.2863,9.9559,99.4860,485576369.0,0,normal
2,2024-01-01 00:10:00,URLLC,33.800517,-7.556512,2.4245,0.4927,5.1083,0.7753,5.1232,99.9995,0.0005,0.0005,0.4994,97.0391,98.8266,9.9911,99.4985,490452024.0,0,normal


In [5]:
# Capacity features + lag columns
FEATURE_COLS = [
    "throughput_dl_mbps",
    "throughput_ul_mbps",
    "spectral_efficiency_bps_hz",
    "energy_efficiency_bits_per_joule",
    "bler_percent",
    "reliability_percent",
    "one_way_latency_ms",
]

df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
print(f"Clean rows: {len(df):,}")

Clean rows: 210,528


In [6]:
print("\nBuilding sequences...")

X_raw = df[FEATURE_COLS].values
y_raw = df[TARGET_KPI].shift(-1).values
X_raw, y_raw = X_raw[:-1], y_raw[:-1]

X_all, y_all = [], []
for i in range(SEQ_LEN, len(X_raw)):
    X_all.append(X_raw[i - SEQ_LEN : i, :])
    y_all.append(y_raw[i])

X_seq = np.stack(X_all).astype(np.float32)
y_seq = np.array(y_all, dtype=np.float32).reshape(-1, 1)
print(f"Sequences: {len(X_seq):,} | X: {X_seq.shape} | y: {y_seq.shape}")


Building sequences...
Sequences: 210,515 | X: (210515, 12, 7) | y: (210515, 1)


In [7]:
train_n = int(len(X_seq) * TRAIN_RATIO)
X_train, X_test = X_seq[:train_n], X_seq[train_n:]
y_train, y_test = y_seq[:train_n], y_seq[train_n:]

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, X_train.shape[2])).reshape(X_train.shape)
X_test_scaled  = scaler_X.transform(X_test.reshape(-1, X_test.shape[2])).reshape(X_test.shape)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32, device=DEVICE)
X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32, device=DEVICE)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32, device=DEVICE)
y_test_t  = torch.tensor(y_test_scaled,  dtype=torch.float32, device=DEVICE)

print(f"Train: {len(X_train_t):,} | Test: {len(X_test_t):,} | Features: {X_train_t.shape[2]} | Device: {DEVICE}")

Train: 168,412 | Test: 42,103 | Features: 7 | Device: cpu


In [8]:
# ── MODEL 1: LSTM ──────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# ── MODEL 2: CNN-LSTM ─────────────────────────────────────────────────────────
class CNNLSTMModel(nn.Module):
    def __init__(self, input_size, cnn_out=32, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, cnn_out, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(cnn_out,    cnn_out, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.lstm = nn.LSTM(cnn_out, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        x = self.cnn(x.permute(0, 2, 1)).permute(0, 2, 1)  # CNN then back
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# ── MODEL 3: GRU ───────────────────────────────────────────────────────────────
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

In [9]:
def get_batches(X, y, batch_size):
    for i in range(0, len(X), batch_size):
        yield X[i : i + batch_size], y[i : i + batch_size]


def train_and_evaluate(model, name):
    print(f"\n{'─'*55}\n  Training: {name}\n{'─'*55}")
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    loss_fn   = nn.L1Loss()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for bx, by in get_batches(X_train_t, y_train_t, BATCH_SIZE):
            optimizer.zero_grad()
            loss = loss_fn(model(bx), by)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            preds = scaler_y.inverse_transform(model(X_test_t).cpu().numpy())
            true  = scaler_y.inverse_transform(y_test_t.cpu().numpy())
            mae   = mean_absolute_error(true, preds)
            rmse  = math.sqrt(mean_squared_error(true, preds))

        mean_train = np.mean(train_losses)
        scheduler.step(mean_train)
        print(f"  Epoch {epoch}/{EPOCHS} | Train MAE: {mean_train:.4f} | Test MAE: {mae:.4f} | RMSE: {rmse:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    acc = np.mean(np.abs(true - preds) <= TOLERANCE) * 100
    return {"name": name, "model": model, "MAE": mae, "RMSE": rmse, "Accuracy": acc, "preds": preds, "true": true}

In [10]:
input_size = X_train_t.shape[2]
results = []
results.append(train_and_evaluate(LSTMModel(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE),  "LSTM"))
results.append(train_and_evaluate(CNNLSTMModel(input_size).to(DEVICE),                        "CNN-LSTM"))
results.append(train_and_evaluate(GRUModel(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE),   "GRU"))


───────────────────────────────────────────────────────
  Training: LSTM
───────────────────────────────────────────────────────
  Epoch 1/20 | Train MAE: 0.3831 | Test MAE: 4.0039 | RMSE: 7.2911 | LR: 0.001000
  Epoch 2/20 | Train MAE: 0.2700 | Test MAE: 3.4807 | RMSE: 6.0931 | LR: 0.001000
  Epoch 3/20 | Train MAE: 0.2548 | Test MAE: 3.4142 | RMSE: 5.8648 | LR: 0.001000
  Epoch 4/20 | Train MAE: 0.2506 | Test MAE: 3.4054 | RMSE: 5.7929 | LR: 0.001000
  Epoch 5/20 | Train MAE: 0.2486 | Test MAE: 3.3776 | RMSE: 5.7137 | LR: 0.001000
  Epoch 6/20 | Train MAE: 0.2473 | Test MAE: 3.3717 | RMSE: 5.6799 | LR: 0.001000
  Epoch 7/20 | Train MAE: 0.2464 | Test MAE: 3.3600 | RMSE: 5.6368 | LR: 0.001000
  Epoch 8/20 | Train MAE: 0.2457 | Test MAE: 3.3504 | RMSE: 5.6041 | LR: 0.001000
  Epoch 9/20 | Train MAE: 0.2451 | Test MAE: 3.3511 | RMSE: 5.5872 | LR: 0.001000
  Epoch 10/20 | Train MAE: 0.2447 | Test MAE: 3.3422 | RMSE: 5.5563 | LR: 0.001000
  Epoch 11/20 | Train MAE: 0.2442 | Test MAE: 3.3

In [11]:
print("\n" + "═"*62)
print(f"  MODEL COMPARISON  —  Target: {TARGET_KPI}")
print("═"*62)
print(f"  {'Model':<10} {'MAE':>10} {'RMSE':>10} {'Acc (±{:.1f})'.format(TOLERANCE):>14}")
print("  " + "─"*56)
best = min(results, key=lambda r: r["MAE"])
for r in results:
    marker = " ← BEST" if r["name"] == best["name"] else ""
    print(f"  {r['name']:<10} {r['MAE']:>10.4f} {r['RMSE']:>10.4f} {r['Accuracy']:>13.2f}%{marker}")
print("═"*62)


══════════════════════════════════════════════════════════════
  MODEL COMPARISON  —  Target: throughput_dl_mbps
══════════════════════════════════════════════════════════════
  Model             MAE       RMSE     Acc (±5.0)
  ────────────────────────────────────────────────────────
  LSTM           3.3045     5.4436         80.20%
  CNN-LSTM       3.3022     5.4350         80.02% ← BEST
  GRU            3.3173     5.4620         80.10%
══════════════════════════════════════════════════════════════


In [12]:
# === PREDICTION TEST ===
# Test the best model on 4 sample rows from the test set
# Each row = a sequence of SEQ_LEN=12 steps -> predicts the next value of throughput_dl_mbps

import pandas as pd
import torch
import numpy as np

sample_indices = [0, len(X_test)//3, 2*len(X_test)//3, len(X_test)-1]

best_model = best["model"]
best_model.eval()

rows = []
with torch.no_grad():
    for idx in sample_indices:
        seq_raw    = X_test[idx]
        seq_scaled = scaler_X.transform(seq_raw).astype(np.float32)
        seq_t      = torch.tensor(seq_scaled[np.newaxis, :, :], device=DEVICE)

        pred_scaled = best_model(seq_t).cpu().numpy()
        pred_val    = scaler_y.inverse_transform(pred_scaled)[0, 0]
        true_val    = float(y_test[idx, 0])
        error       = abs(pred_val - true_val)
        within      = "✓" if error <= TOLERANCE else "✗"

        rows.append({
            "Sample index":                                    idx,
            f"Last known {TARGET_KPI}":                     round(seq_raw[-1, FEATURE_COLS.index(TARGET_KPI)], 4),
            "Predicted (t+1)":                                round(pred_val, 4),
            "Actual (t+1)":                                   round(true_val, 4),
            "Abs Error":                                      round(error, 4),
            f"Within ±{TOLERANCE} Mbps":                 "✓" if error <= TOLERANCE else "✗",
        })

df_test = pd.DataFrame(rows)
print(f"Best model  : {best['name']}")
print(f"Target KPI  : {TARGET_KPI}")
print(f"Tolerance   : ±{TOLERANCE} Mbps")
print()
display(df_test)


Best model  : CNN-LSTM
Target KPI  : throughput_dl_mbps
Tolerance   : ±5.0 Mbps



,Sample index,Last known throughput_dl_mbps,Predicted (t+1),Actual (t+1),Abs Error,Within ±5.0 Mbps
0,0,107.482002,107.459602,102.4033,5.0564,✗
1,14034,92.365097,95.327003,96.8118,1.4848,✓
2,28068,17.444099,34.144501,38.0535,3.9090,✓
3,42102,96.767998,98.029900,95.1023,2.9276,✓


In [13]:
# =====================================================================
# Export best model + scalers to PKL_models/
# =====================================================================
import os, joblib

pkl_dir = os.path.normpath(os.path.join(os.getcwd(), "..", "kafka_streaming", "PKL_models"))
os.makedirs(pkl_dir, exist_ok=True)

best_model = best["model"]
best_model.eval()

# Save PyTorch model (full object for easy reload)
model_path = os.path.join(pkl_dir, f"{TARGET_KPI}_best_model.pt")
torch.save(best_model, model_path)
print(f"Model  saved : {model_path}")

# Save scalers
scaler_X_path = os.path.join(pkl_dir, f"{TARGET_KPI}_scaler_X.pkl")
scaler_y_path = os.path.join(pkl_dir, f"{TARGET_KPI}_scaler_y.pkl")
joblib.dump(scaler_X, scaler_X_path)
joblib.dump(scaler_y, scaler_y_path)
print(f"scaler_X saved : {scaler_X_path}")
print(f"scaler_y saved : {scaler_y_path}")

# Save feature column list
feat_path = os.path.join(pkl_dir, f"{TARGET_KPI}_feature_cols.pkl")
joblib.dump(FEATURE_COLS, feat_path)
print(f"Feature cols saved : {feat_path}")


Model  saved : /Users/ayoubkallel/PFA2/-Intelligent-Anomaly-Monitoring-in-5G-networks/kafka_streaming/PKL_models/throughput_dl_mbps_best_model.pt
scaler_X saved : /Users/ayoubkallel/PFA2/-Intelligent-Anomaly-Monitoring-in-5G-networks/kafka_streaming/PKL_models/throughput_dl_mbps_scaler_X.pkl
scaler_y saved : /Users/ayoubkallel/PFA2/-Intelligent-Anomaly-Monitoring-in-5G-networks/kafka_streaming/PKL_models/throughput_dl_mbps_scaler_y.pkl
Feature cols saved : /Users/ayoubkallel/PFA2/-Intelligent-Anomaly-Monitoring-in-5G-networks/kafka_streaming/PKL_models/throughput_dl_mbps_feature_cols.pkl


In [35]:
# === EXPORT ALL PREDICTIONS FOR POWER BI ===
best_model = best["model"]
best_model.eval()

predictions = []

with torch.no_grad():
    for idx in range(len(X_test)):
        seq_raw = X_test[idx]
        seq_scaled = scaler_X.transform(seq_raw).astype(np.float32)
        seq_t = torch.tensor(seq_scaled[np.newaxis, :, :], device=DEVICE)

        pred_scaled = best_model(seq_t).cpu().numpy()
        pred_val = float(scaler_y.inverse_transform(pred_scaled)[0, 0])
        true_val = float(y_test[idx, 0])

        mae = best["MAE"]
        row_idx = train_n + SEQ_LEN + idx

        predictions.append({
            "timestamp": str(df.iloc[row_idx]["timestamp"]),
            "slice_type": df.iloc[row_idx]["slice_type"],
            "kpi_name": TARGET_KPI,
            "actual_value": round(true_val, 4),
            "predicted_value": round(pred_val, 4),
            "confidence_upper": round(pred_val + 1.5 * mae, 4),
            "confidence_lower": round(max(0, pred_val - 1.5 * mae), 4),
            "model_name": best["name"]
        })

df_pred = pd.DataFrame(predictions)
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_pred.to_csv(f"{OUTPUT_DIR}/{TARGET_KPI}_predictions.csv", index=False)
print(f"Exported {len(df_pred)} predictions to {OUTPUT_DIR}/{TARGET_KPI}_predictions.csv")
df_pred.head()

Exported 42103 predictions to ../outputs/capacity/throughput_dl_mbps_predictions.csv


,timestamp,slice_type,kpi_name,actual_value,predicted_value,confidence_upper,confidence_lower,model_name
0,2025-08-07 19:20:00,URLLC,throughput_dl_mbps,102.4033,107.4597,112.4130,102.5063,CNN-LSTM
1,2025-08-07 19:25:00,URLLC,throughput_dl_mbps,103.7566,102.6221,107.5754,97.6687,CNN-LSTM
2,2025-08-07 19:30:00,URLLC,throughput_dl_mbps,99.8713,102.9641,107.9174,98.0108,CNN-LSTM
3,2025-08-07 19:35:00,URLLC,throughput_dl_mbps,102.4115,104.0400,108.9933,99.0867,CNN-LSTM
4,2025-08-07 19:40:00,URLLC,throughput_dl_mbps,103.4069,101.1170,106.0703,96.1637,CNN-LSTM
